# Train a ball detector for Match Video Tracker

The standard YOLO model learned "sports ball" from everyday photos, so it finds the small, blurred
ball in a broadcast frame only about 30 % of the time. Here we **fine-tune** YOLO on football
footage: start from a model that already knows how to find objects, and show it about 5,000
labelled broadcast pictures of the ball.

**Data:** Roboflow's free [football-ball-detection](https://universe.roboflow.com/roboflow-jvuqo/football-ball-detection-rejhg)
set (version 4, licence CC BY 4.0, Bundesliga broadcast frames, 1 class: `ball`). Each frame was cut into
4 tiles, so the ball looks bigger. `m5_possession.py` looks at your video in 4 tiles the same way.

## Before you press Run

1. **Runtime → Change runtime type → T4 GPU → Save.** Training on a CPU would take days.
2. Make a free account at [roboflow.com](https://roboflow.com), then **Settings → API Keys** and copy your private key.
3. In Colab, click the **key icon** in the left bar → **Add new secret** → name `ROBOFLOW_API_KEY`, paste the key, and switch on **Notebook access**.
4. **Runtime → Run all.** Training takes about an hour. Keep the tab open.
5. At the end your browser downloads `ball.pt`. Move it into your `football-cv` folder (next to `yolo26m.pt`).

## 1. Check the GPU
You should see a Tesla T4. If you see an error, do step 1 above.

In [ ]:
!nvidia-smi

## 2. Install Ultralytics (YOLO) and Roboflow

In [ ]:
%pip install -q ultralytics roboflow

## 3. Download the dataset
About 5,000 pictures with a box drawn around every ball.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace("roboflow-jvuqo").project("football-ball-detection-rejhg").version(4).download("yolov8")
print("Saved in", dataset.location)

## 4. Point the dataset file at the right folders
Roboflow's `data.yaml` uses relative paths that Ultralytics sometimes can't follow, so we write full ones.

In [ ]:
import os, yaml

data_file = os.path.join(dataset.location, "data.yaml")
with open(data_file) as f:
    data = yaml.safe_load(f)
data["path"] = dataset.location
data["train"], data["val"], data["test"] = "train/images", "valid/images", "test/images"
with open(data_file, "w") as f:
    yaml.safe_dump(data, f)
print(data)
for part in ("train", "valid", "test"):
    print(part, len(os.listdir(os.path.join(dataset.location, part, "images"))), "pictures")

## 5. Train
- `yolo26s.pt`: the small YOLO26, already trained on COCO. Fine-tuning starts from what it knows.
- `epochs=50`: it goes through all the pictures 50 times. Each epoch takes about a minute on a T4.
- `imgsz=640`: the size of each tile, the same size `m5_possession.py` uses.
- `patience=15`: stop early if 15 epochs in a row bring no improvement.

The `mAP50` column is the score to watch: the share of balls it finds with a well-placed box. Roboflow's own model reached about 0.9.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
results = model.train(data=data_file, epochs=50, imgsz=640, batch=16, patience=15,
                      project="runs", name="ball", plots=True)

## 6. How good is it?
Precision = how often a box really is a ball. Recall = how many of the balls it finds.

In [ ]:
metrics = YOLO("runs/ball/weights/best.pt").val(data=data_file, split="test")
print(f"precision {metrics.box.mp:.2f}   recall {metrics.box.mr:.2f}   mAP50 {metrics.box.map50:.2f}")

## 7. Look at a few test pictures
The boxes should sit on the ball, even when it is small or next to a boot.

In [ ]:
import glob
from IPython.display import Image, display

best = YOLO("runs/ball/weights/best.pt")
pictures = sorted(glob.glob(os.path.join(dataset.location, "test/images/*")))[:4]
for r in best.predict(pictures, conf=0.25, save=True, project="runs", name="check", exist_ok=True, verbose=False):
    pass
for path in sorted(glob.glob("runs/check/*"))[:4]:
    display(Image(filename=path, width=640))

## 8. Download `ball.pt`
Move it into your `football-cv` folder, then run `python m5_possession.py data/clip1.mov` on your Mac.

In [ ]:
import shutil
from google.colab import files

shutil.copy("runs/ball/weights/best.pt", "ball.pt")
files.download("ball.pt")